In [1]:
import transformers
from datasets import load_dataset

# instantiate the model
name = "answerdotai/ModernBERT-base"
model = transformers.AutoModelForSequenceClassification.from_pretrained(
    name,
    dtype="auto",
    num_labels=2,
)

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [2]:
# set up the dataset and dataset loading
tokenizer = transformers.AutoTokenizer.from_pretrained(name)
ds_raw = load_dataset("stanfordnlp/sst2")
ds = ds_raw.map(
    lambda item: tokenizer(
        item["sentence"],
        truncation=True,
        max_length=512,
    )
)

# batch collator
collator = transformers.DataCollatorWithPadding(tokenizer)


tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

...

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [ ]:
# set up training params
for param in model.base_model.parameters():
    param.requires_grad_(False)

training_args = transformers.TrainingArguments(
    output_dir="./ModernBert",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    learning_rate=2e-5,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    load_best_model_at_end=True,
)

trainer = transformers.Trainer(
    model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tokenizer,
    data_collator=collator,
)
trainer.train()

Step,Training Loss,Validation Loss
1000,4.487362,0.522142


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]